In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [48]:
from dataclasses import dataclass

from langchain.agents import  create_agent
from langchain_openrouter import ChatOpenRouter

from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call, AgentMiddleware, AgentState

import time
from langchain.messages import SystemMessage, HumanMessage, AIMessage

In [21]:
basic_model = ChatOpenRouter(
    # model="nvidia/nemotron-3-ultra-550b-a55b:free",
    model="dots-studio/dots-3-note-preview:free",
    temperature=0.2,
)

advance_model = ChatOpenRouter(
    # model="google/gemma-4-31b-it:free",
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    temperature=0.2,
)

In [36]:

@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    message_count = len(request.state["messages"])

    if message_count > 3:
        model = advance_model
    else:
        model = basic_model

    # DEBUG: Print which model is selected
    print("Message count:", message_count)
    print("Selected model:", model)

    request.model = model
    return handler(request)
    # return handler(request.override(model=model))


In [37]:
agent =create_agent(
    model=basic_model,
    middleware=[dynamic_model_selection]
)

In [ ]:

# try:
#     response = agent.invoke({
#         "messages": [
#             SystemMessage("You are a helpful assistant."),
#             HumanMessage("What is 1+1?")
#         ]
#     })

#     print(response)

# except Exception as e:
#     print("Exception type:", type(e).__name__)
#     print("Error message:", str(e))
#     print("Error details:", repr(e))

#     for attr in ("status_code", "body", "response"):
#         value = getattr(e, attr, None)
#         if value is not None:
#             print(f"{attr}:", value)


Exception type: UnboundLocalError
Error message: cannot access local variable 'model' where it is not associated with a value
Error details: UnboundLocalError("cannot access local variable 'model' where it is not associated with a value")


In [41]:
response = agent.invoke({
    'messages' : [
        SystemMessage('You are a helpful assistant.'),
        HumanMessage('What is 1+1?'),
        HumanMessage('What is 1+1?'),
        HumanMessage('What is 1+1?'),
        HumanMessage('What is 1+1?'),
        HumanMessage('What is 1+1?')
    ]
})

Message count: 6
Selected model: metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.6', 'langchain-openrouter': '0.2.9'}} profile={'name': 'Nemotron 3 Ultra (free)', 'release_date': '2026-06-04', 'last_updated': '2026-06-04', 'open_weights': True, 'max_input_tokens': 1000000, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': False, 'attachment': False, 'temperature': True, 'tool_call_streaming': True} client=<openrouter.sdk.OpenRouter object at 0x110e342c0> openrouter_api_key=SecretStr('**********') app_url='https://docs.langchain.com' app_title='LangChain' model_name='nvidia/nemotron-3-ultra-550b-a55b:free' temperature=0.2 model_kwargs={}


/var/folders/hm/m6bjtvn14_v4wbn68_y2tq980000gn/T/ipykernel_58937/226581060.py:14: DeprecationWarning: Direct attribute assignment to ModelRequest.model is deprecated. Use request.override(model=...) instead to create a new request with the modified attribute.
  request.model = model


In [42]:
print(response['messages'][-1].content)

1 + 1 = 2


In [43]:
print(response['messages'][-1].response_metadata['model_name'])

nvidia/nemotron-3-ultra-550b-a55b:free


In [61]:
class HooksDemo(AgentMiddleware):
    
    def __init__(self):
        super().__init__()
        self.start_time = 0.0
        
    def before_agent(self, state: AgentState, runtime):
        self.start_time = time.time()
        print('before_agent triggered')
        
    def before_model(self, state: AgentState, runtime):
        print('before_model triggered')
        
    def after_model(self, state: AgentState, runtime):
        print('after_model triggered')
        
    def after_agent(self, state: AgentState, runtime):
        print('after_agent:', time.time()-self.start_time)
    

In [62]:
model = ChatOpenRouter(
    model="openrouter/free",
    temperature=0.2,
)

In [63]:
agent =create_agent(
    model=model,
    middleware=[HooksDemo()]
)

In [64]:
response = agent.invoke({
    'messages' : [
        SystemMessage('You are a helpful assistant.'),
        HumanMessage('What is PCA?')
    ]
})

before_agent triggered
before_model triggered
after_model triggered
after_agent: 35.8195059299469


In [65]:
print(response['messages'][-1].content)

**PCA (Principal Component Analysis)** is a widely‑used statistical technique for reducing the dimensionality of a dataset while preserving as much of its variability (information) as possible.  

---

## 1. The intuition

Imagine you have a table of measurements (features) for many observations (samples).  
Some columns are highly correlated—e.g., height and weight of the same people.  
If you keep all columns, you’re using a lot of redundancy.  

PCA finds a new set of orthogonal (uncorrelated) axes called **principal components (PCs)** that capture the directions in which the data varies the most. By projecting the original data onto the top k PCs, you obtain a lower‑dimensional representation that retains most of the original information.

---

## 2. Formal definition

Given a data matrix  

\[
\mathbf{X} = \begin{bmatrix}
\mathbf{x}_1^\top\\
\mathbf{x}_2^\top\\
\vdots\\
\mathbf{x}_n^\top
\end{bmatrix}\in\mathbb{R}^{n\times p}
\]

where each row \(\mathbf{x}_i\) is a p‑dimensional 